# Обучение YOLO на GPU (Google Colab)

**Сначала включи GPU** (один раз):
`Среда выполнения → Сменить среду выполнения → Графический процессор T4 → Сохранить`

Дальше жми кнопку ▶ слева от каждой ячейки, по порядку:

1. Ячейка 1 — загрузи `aim_colab.zip` (появится кнопка «Выбрать файлы»).
2. Ячейка 2 — поставит ultralytics и проверит GPU.
3. Ячейка 3 — само обучение (50 эпох, идёт несколько минут).
4. Ячейка 4 — скачает готовый `best.onnx` в «Загрузки».


In [ ]:
from google.colab import files
import zipfile, shutil, os

uploaded = files.upload()   # выбери файл aim_colab.zip
name = list(uploaded.keys())[0]

with open('/content/aim_colab.zip', 'wb') as f:
    f.write(uploaded[name])

shutil.rmtree('/content/aim', ignore_errors=True)
with zipfile.ZipFile('/content/aim_colab.zip') as z:
    z.extractall('/content/aim')

print('Распаковано ->', sorted(os.listdir('/content/aim')))


In [ ]:
!pip install -q ultralytics onnx onnxslim
import torch
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! GPU не включён: Среда выполнения → Сменить среду → T4 → Сохранить')


In [ ]:
%cd /content/aim/aim_dataset
!python train.py --model yolov8n.pt --epochs 80 --patience 25 --imgsz 640 --batch 16 --device 0


In [ ]:
import glob
from google.colab import files

onnx = sorted(glob.glob('/content/aim/aim_dataset/runs/detect/*/weights/best.onnx'))
if onnx:
    files.download(onnx[-1])
    print('Скачивается в «Загрузки» — подложи рядом с capture_demo.exe')
else:
    print('best.onnx не найден — обучение не завершилось без ошибок')
